In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, regexp_extract, to_date, col
from pyspark.sql.types import *

In [0]:
source_path = "s3://subscription-revenue-platform/raw/subscription_events/"

bronze_table_path = "s3://subscription-revenue-platform/bronze/subscription_events/"
bronze_checkpoint_path = "s3://subscription-revenue-platform/bronze/bronze_subscription_events_checkpoints/"
bronze_schema_path = "s3://subscription-revenue-platform/bronze/bronze_subscription_events_schema/"

In [0]:
catalog = "subscription_platform"
schema = "bronze"
bronze_table = f"{catalog}.{schema}.subscription_events"

In [0]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("customer_id", StringType()),
    StructField("subscription_id", StringType()),
    StructField("plan_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("currency", StringType()),
    StructField("event_time", TimestampType()),
    StructField("ingested_at", TimestampType()),
    StructField("source", StringType())
])

In [0]:
raw_events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", bronze_schema_path)
    .schema(event_schema)
    .load(source_path)
)

In [0]:
bronze_events = (
    raw_events
    .withColumn("processed_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
(
    bronze_events.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

In [0]:
%sql

SELECT *
FROM subscription_platform.bronze.subscription_events
ORDER BY processed_at DESC
LIMIT 20

event_id,event_type,customer_id,subscription_id,plan_id,amount,currency,event_time,ingested_at,source,processed_at,source_file
evt_9ecbe4fe4205461685b0395a1e8871ba,SUBSCRIPTION_CANCELLED,cust_507,sub_1007,basic,1000.0,INR,2026-04-10T21:20:04.810Z,2026-04-10T21:20:04.810Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_9ecbe4fe4205461685b0395a1e8871ba.json
evt_bbadb252f1464c8cb8aaf8f5ff93c339,SUBSCRIPTION_REACTIVATED,cust_504,sub_1004,basic,1000.0,INR,2026-04-09T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_bbadb252f1464c8cb8aaf8f5ff93c339.json
evt_e39e4ac7160f435caa98c89a168b29a8,SUBSCRIPTION_CANCELLED,cust_508,sub_1008,pro,2000.0,INR,2026-04-09T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_e39e4ac7160f435caa98c89a168b29a8.json
evt_2c3e14a1caeb411cb4c9794fbcbbe8c6,SUBSCRIPTION_REACTIVATED,cust_509,sub_1009,basic,1000.0,INR,2026-04-10T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_2c3e14a1caeb411cb4c9794fbcbbe8c6.json
evt_ad81ac159a204c83b79a445c3a74f185,SUBSCRIPTION_REACTIVATED,cust_502,sub_1002,pro,2000.0,INR,2026-04-10T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_ad81ac159a204c83b79a445c3a74f185.json
evt_2e2a1510bf2a4ff79d8a8594252d48e0,SUBSCRIPTION_CANCELLED,cust_508,sub_1008,pro,2000.0,INR,2026-04-10T21:19:58.887Z,2026-04-10T21:19:58.887Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_2e2a1510bf2a4ff79d8a8594252d48e0.json
evt_9e837621a3ae4a54bd39664eef06ee88,PLAN_CHANGED,cust_505,sub_1005,basic,1000.0,INR,2026-04-10T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_9e837621a3ae4a54bd39664eef06ee88.json
evt_5c02ea6200a449c0bbf6e65f676ed98f,SUBSCRIPTION_CREATED,cust_508,sub_1008,pro,2000.0,INR,2026-04-07T21:19:58.887Z,2026-04-10T21:19:58.887Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-07/evt_5c02ea6200a449c0bbf6e65f676ed98f.json
evt_943ffada3ed44047a1199bb778173deb,SUBSCRIPTION_CREATED,cust_506,sub_1006,pro,2000.0,INR,2026-04-10T21:20:04.810Z,2026-04-10T21:20:04.810Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-10/evt_943ffada3ed44047a1199bb778173deb.json
evt_187b9f36644e472eb767d15ce82dca4b,PLAN_CHANGED,cust_504,sub_1004,basic,1000.0,INR,2026-04-09T21:20:02.202Z,2026-04-10T21:20:02.202Z,billing_lambda,2026-04-10T21:20:40.569Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_187b9f36644e472eb767d15ce82dca4b.json


In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM subscription_platform.bronze.subscription_events;

total_records
98


In [0]:
%sql

SELECT
    event_type,
    COUNT(*) AS event_count
FROM subscription_platform.bronze.subscription_events
GROUP BY event_type
ORDER BY event_count DESC;

event_type,event_count
SUBSCRIPTION_REACTIVATED,31
SUBSCRIPTION_CREATED,27
PLAN_CHANGED,23
SUBSCRIPTION_CANCELLED,17


In [0]:
%sql

SELECT
    MIN(event_time) AS earliest_event,
    MAX(event_time) AS latest_event
FROM subscription_platform.bronze.subscription_events;

earliest_event,latest_event
2026-04-06T18:04:14.732Z,2026-04-10T21:20:04.810Z


In [0]:
%sql

SELECT
    source_file,
    COUNT(*) AS records
FROM subscription_platform.bronze.subscription_events
GROUP BY source_file
ORDER BY records DESC;


source_file,records
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_cc3668af98604eab8d47c59142e8fff8.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_b8fdfe0b0d1e49128fab8bbe87c17b6b.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-08/evt_a86918fcbaf4441c9264a53f8c9a3e89.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-08/evt_29fce323a291495c8c23319ead8f3308.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_02ef60d02098455d810b15293bba92ea.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_1e02a3c2001d44dfa8cc30fad2cd86c8.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-06/evt_43fb61ec1b6f4121a60452916bfebab4.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_15673b9f9d5b48a0bf1f530e26e2701e.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_404316929a924ea699996616c56fb71c.json,1
s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-08/evt_0cfb749fc97f46dd9e191bb20c80d8c1.json,1
